# Geometry Working Memory

https://www.biorxiv.org/content/10.64898/2026.08.31.748237v1

In [ ]:
# libraries

import os
import numpy as np
import pickle

In [10]:
# functions


def compute_euclidean_distances(points):
    """
    Computes the pairwise Euclidean distances between all points in R^n space.

    Parameters:
    points (np.ndarray): An array of shape (num_points, n), where each row represents a point in space of n dimensions

    Returns:
    distances (np.ndarray): A matrix of pairwise Euclidean distances between all points.
    """
    num_points = points.shape[0]
    distances = np.zeros((num_points, num_points))

    for i in range(num_points):
        for j in range(i + 1, num_points):
            # Compute the Euclidean distance between points i and j
            dist = np.linalg.norm(points[i] - points[j])
            distances[i, j] = dist
            distances[j, i] = dist  # Distance is symmetric

    return distances


def extract_upper_triangle(matrix):
    """
    Extracts all the values above the diagonal (upper triangle without the diagonal elements) of a square matrix.

    Parameters:
    matrix (np.ndarray): A square matrix (n x n).

    Returns:
    np.ndarray: A 1D array containing all values above the diagonal.
    """
    # Ensure the matrix is square
    if matrix.shape[0] != matrix.shape[1]:
        raise ValueError("The input matrix must be square.")

    # Extract the upper triangle without the diagonal
    upper_triangle = matrix[np.triu_indices_from(matrix, k=1)]

    return upper_triangle




In [ ]:
### set paths and settings

path_root = '/path_to_local'

# settings

subjects = [f'sub_{i:02d}' for i in range(1, 50)]

time_windows = ['encode', 'maint', 's2']
trial_type = 'correct_trials'
pca_aligned_folder = 'pca_aligned'
k_start, k_end = 0, 2       # 0,2 means leading three PCs
trial_types = ['correct_trials', 'incorrect_trials']

outputs_correct_trials = [
'control_2gratings_2polygons',                                  'control_1gratings_1polygons',
'update_2gratings_relevant_2polygons_nonrelevant',              'update_1gratings_relevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_relevant',              'update_1gratings_nonrelevant_1polygons_relevant',
'inhibition_2gratings_relevant_2polygons_nonrelevant',          'inhibition_1gratings_relevant_1polygons_nonrelevant',
'inhibition_2gratings_nonrelevant_2polygons_relevant',          'inhibition_1gratings_nonrelevant_1polygons_relevant',
'update_2gratings_nolongerrelevant_2polygons_nonrelevant',      'update_1gratings_nolongerrelevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_nolongerrelevant',      'update_1gratings_nonrelevant_1polygons_nolongerrelevant',
]

outputs_incorrect_trials = [
'control_2gratings_2polygons',                                  
'update_2gratings_relevant_2polygons_nonrelevant',              
'update_2gratings_nonrelevant_2polygons_relevant',              
'inhibition_2gratings_relevant_2polygons_nonrelevant',          
'inhibition_2gratings_nonrelevant_2polygons_relevant',        
'update_2gratings_nolongerrelevant_2polygons_nonrelevant',      
'update_2gratings_nonrelevant_2polygons_nolongerrelevant',       
]


# Separability - euclidean distance

In [12]:
for trial_type in trial_types:

    if trial_type == 'correct_trials':
        outputs = outputs_correct_trials
    elif trial_type == 'incorrect_trials':
        outputs = outputs_incorrect_trials

    for time_window in time_windows:

        path_outputs = os.path.join(path_root, 'results', pca_aligned_folder, time_window + '_time_resolved', 'allsubjects', trial_type)

        if not os.path.isdir(path_outputs):
            os.makedirs(path_outputs)
    
        if not os.path.isfile(os.path.join(path_outputs, 'edist_' + time_window + '_shape_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl')):

            if time_window == 'encode':

                segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

            elif time_window == 'maint':

                segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

            elif time_window == 's2':

                segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

            edist_orientation = {}
            edist_shape = {}

            # loop over conditions (including stimulus load)
            for output_idx, output_i in enumerate(outputs):

                # empty matrix for outputs
                matrix_edist_orientation = np.empty((len(subjects), len(segments)))
                matrix_edist_shape = np.empty((len(subjects), len(segments)))

                for sub_idx, sub_i in enumerate(subjects):

                    # load file with pca outputs
                    file = os.path.join(path_root, 'results', pca_aligned_folder, time_window + '_time_resolved', sub_i, 'pca_' + trial_type + '.pkl')
                    with open(file, 'rb') as file:
                        data = pickle.load(file)

                    for time_idx, time_segment in enumerate(segments):

                        key = time_window + '_segment' + str(time_segment) + '_' + output_i

                        pc_scores = data['pc_scores_' + key]
                        pc_scores = pc_scores[:,k_start:(k_end+1)]
                        distances_orientation = compute_euclidean_distances(pc_scores[0:4,:]) # orientation
                        matrix_edist_orientation[sub_idx, time_idx] = np.mean(extract_upper_triangle(distances_orientation))
                        distances_shape = compute_euclidean_distances(pc_scores[4:7,:]) # shape
                        matrix_edist_shape[sub_idx, time_idx] = np.mean(extract_upper_triangle(distances_shape))
                        
                # assign to dict
                key_orientation = time_window + '_edistorient_' + output_i
                edist_orientation[key_orientation] = matrix_edist_orientation

                key_shape = time_window + '_edistshape_' + output_i
                edist_shape[key_shape] = matrix_edist_shape

            # save            
            with open(os.path.join(path_outputs, 'edist_' + time_window + '_orientation_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
                pickle.dump(edist_orientation, file)

            with open(os.path.join(path_outputs, 'edist_' + time_window + '_shape_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
                pickle.dump(edist_shape, file)
